<a href="https://colab.research.google.com/github/miaouiAmine/NLP-Translation-Demo/blob/main/Minimal_Seq2Seq_Translator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Embedding, Dense

In [ ]:
# Training Data
english_texts = [
    'Hello!',
    'Good morning!',
    'How are you?',
    'I love programming.',
    'What is your name?',
    'The weather is nice today.',
    'Where is the library?',
    'Thank you very much.',
    'Can you help me?',
    'Have a great day!'
]

french_texts = [
    'Bonjour !',
    'Bonjour le matin !',  # Note: Actual French would typically just use "Bonjour"
    'Comment allez-vous?',
    'J adore la programmation.',
    'Quel est votre nom?',
    'Il fait beau aujourd hui.',
    'Où est la bibliothèque?',
    'Merci beaucoup.',
    'Pouvez-vous m aider?',
    'Passez une excellente journee!'
]

In [ ]:
# Preprocessing
# Add start/end tokens to French sentences
french_texts = ['<start> ' + text + ' <end>' for text in french_texts]

In [ ]:
# Tokenization
tokenizer_eng = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')
tokenizer_fr = Tokenizer(filters='!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n')

In [ ]:
tokenizer_eng.fit_on_texts(english_texts)
tokenizer_fr.fit_on_texts(french_texts)

In [ ]:
# Convert texts to sequences
eng_sequences = tokenizer_eng.texts_to_sequences(english_texts)
fr_sequences = tokenizer_fr.texts_to_sequences(french_texts)

In [ ]:
# Padding
max_eng_len = max(len(seq) for seq in eng_sequences)
max_fr_len = max(len(seq) for seq in fr_sequences)

In [ ]:
encoder_inputs = pad_sequences(eng_sequences, maxlen=max_eng_len, padding='post')
decoder_inputs = pad_sequences([seq[:-1] for seq in fr_sequences], maxlen=max_fr_len-1, padding='post')
decoder_targets = pad_sequences([seq[1:] for seq in fr_sequences], maxlen=max_fr_len-1, padding='post')

In [ ]:
# Vocabulary sizes
eng_vocab_size = len(tokenizer_eng.word_index) + 1
fr_vocab_size = len(tokenizer_fr.word_index) + 1

In [ ]:
# Model Architecture
# Encoder
encoder_inputs_layer = Input(shape=(max_eng_len,))
enc_emb = Embedding(eng_vocab_size, 256)(encoder_inputs_layer)
encoder_lstm = LSTM(256, return_state=True)
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

In [ ]:
# Decoder
decoder_inputs_layer = Input(shape=(max_fr_len-1,))
dec_emb = Embedding(fr_vocab_size, 256)(decoder_inputs_layer)
decoder_lstm = LSTM(256, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=encoder_states)
decoder_dense = Dense(fr_vocab_size, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

In [ ]:
# Full model
model = Model([encoder_inputs_layer, decoder_inputs_layer], decoder_outputs)

In [ ]:
# Compile & Train
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [ ]:
# Split data (8 train, 2 test)
train_idx = [0,1,2,3,4,5,6,7]
test_idx = [8,9]

In [ ]:
# Training data
train_enc = encoder_inputs[train_idx]
train_dec_in = decoder_inputs[train_idx]
train_dec_tar = decoder_targets[train_idx]

In [ ]:
# Test data
test_enc = encoder_inputs[test_idx]
test_dec_in = decoder_inputs[test_idx]
test_dec_tar = decoder_targets[test_idx]

In [ ]:
# Training
history = model.fit(
    [train_enc, train_dec_in],
    train_dec_tar,
    validation_data=([test_enc, test_dec_in], test_dec_tar),
    batch_size=2,
    epochs=100,
    verbose=1
)

Epoch 1/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 240ms/step - accuracy: 0.2389 - loss: 3.4789 - val_accuracy: 0.1667 - val_loss: 3.4659
Epoch 2/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.2472 - loss: 3.3375 - val_accuracy: 0.1667 - val_loss: 3.3920
Epoch 3/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.2222 - loss: 3.0177 - val_accuracy: 0.1667 - val_loss: 3.1131
Epoch 4/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.2333 - loss: 2.5380 - val_accuracy: 0.1667 - val_loss: 3.1119
Epoch 5/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.4250 - loss: 2.0618 - val_accuracy: 0.3333 - val_loss: 3.0897
Epoch 6/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.3889 - loss: 2.3174 - val_accuracy: 0.3333 - val_loss: 3.0397
Epoch 7/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.4306 - loss: 2.1140 - val_accuracy: 0.1667 - val_loss: 3.1399
Epoch 8/100
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.4111 - loss: 2.0097 - val_accuracy: 0.1667 - val_loss

In [ ]:
# Inference Setup
# Encoder model
encoder_model = Model(encoder_inputs_layer, encoder_states)

In [ ]:
# Decoder model
decoder_state_input_h = Input(shape=(256,))
decoder_state_input_c = Input(shape=(256,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

In [ ]:
dec_emb2 = Embedding(fr_vocab_size, 256)(decoder_inputs_layer)
decoder_outputs2, state_h2, state_c2 = decoder_lstm(dec_emb2, initial_state=decoder_states_inputs)
decoder_states2 = [state_h2, state_c2]
decoder_outputs2 = decoder_dense(decoder_outputs2)

In [ ]:
decoder_model = Model(
    [decoder_inputs_layer] + decoder_states_inputs,
    [decoder_outputs2] + decoder_states2
)

In [ ]:
# Translation function
def translate(input_text):
    # Preprocess input
    input_seq = tokenizer_eng.texts_to_sequences([input_text])
    input_seq = pad_sequences(input_seq, maxlen=max_eng_len, padding='post')

    # Encode input
    states_value = encoder_model.predict(input_seq)

    # Generate empty target sequence
    target_seq = np.zeros((1, 1))
    target_seq[0, 0] = tokenizer_fr.word_index['<start>']

    decoded_sentence = []
    for _ in range(max_fr_len-1):
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value
        )

        # Get next word
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = tokenizer_fr.index_word.get(sampled_token_index, '')

        if sampled_word == '<end>':
            break

        decoded_sentence.append(sampled_word)

        # Update states and target sequence
        target_seq = np.zeros((1, 1))
        target_seq[0, 0] = sampled_token_index
        states_value = [h, c]

    return ' '.join(decoded_sentence)


In [ ]:
# Test Translations
test_phrases = [
    'Hello!',
    'Good morning!',
    'How are you?',
    'I love programming.',
    'What is your name?',
    'The weather is nice today.',
    'Where is the library?',
    'Thank you very much.',
    'Can you help me?',
    'Have a great day!'
]

for phrase in test_phrases:

    print(f"French: {translate(phrase)}")
    print(f"English: {phrase}")
    print("---")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
French: bonjour
English: Hello!
---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
French: bonjour le matin
English: Good morning!
---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
French: comment allez vous
English: How are you?
---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
French: j adore la programmation
English: I love programming.
---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/